**Install required packages and Imports**

In [7]:
!pip install openpyxl
!pip install biopython
!pip install selenium

import csv
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

**Calculate physicochemical properties**

In [8]:
# ============================================================
# This step:
# 1. Loads the AI prediction output file (.xlsx or .xls)
# 2. Reads protein sequences directly from the "sequence" column
# 3. Retains prediction-related metadata when available
# 4. Calculates physicochemical properties for each sequence
# 5. Saves the results as a CSV file for downstream analysis
#
# NOTE:
# The input file should contain a column named "sequence".
# Additional columns such as predicted_label, confidence_score,
# activity, annotation, or probability can also be retained if present.
# ============================================================

import csv
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Specify the path to the AI prediction output file and output CSV file
input_file = "prediction_output.xlsx"
csv_file = "physicochemical_properties.csv"

# Mapping of one-letter to three-letter amino acid abbreviations
amino_acid_map = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp', 'C': 'Cys',
    'Q': 'Gln', 'E': 'Glu', 'G': 'Gly', 'H': 'His', 'I': 'Ile',
    'L': 'Leu', 'K': 'Lys', 'M': 'Met', 'F': 'Phe', 'P': 'Pro',
    'S': 'Ser', 'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val'
}

# List of standard amino acids (one-letter codes)
amino_acids = list(amino_acid_map.keys())

# Load the AI prediction output
dataset = pd.read_excel(input_file, na_filter=False)

# Ensure that the sequence column is present
if "sequence" not in dataset.columns:
    raise ValueError(
        "The input file must contain a column named 'sequence'."
    )

# Define optional metadata columns to retain if available
candidate_metadata_cols = [
    "annotation",
    "activity",
    "probability"
]

metadata_cols = [col for col in candidate_metadata_cols if col in dataset.columns]

# Open the CSV file for writing
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)

    # Write the header row
    writer.writerow(
        metadata_cols + [
            "Sequence", "Length", "MW", "pI",
            "Instability", "Aromaticity",
            "GRAVY", "Aliphatic", "Helix",
            "Turn", "Sheet"
        ] + [amino_acid_map[aa] for aa in amino_acids]
    )

    # Analyze each sequence in the prediction output
    for idx, row in dataset.iterrows():
        seq = str(row["sequence"]).strip().upper()

        # Skip missing or empty entries
        if seq == "" or seq.lower() == "nan":
            continue

        # Keep only standard amino acids for ProtParam analysis
        cleaned_seq = "".join([aa for aa in seq if aa in amino_acids])

        # Skip sequences that become empty after cleaning
        if len(cleaned_seq) == 0:
            continue

        analysed_seq = ProteinAnalysis(cleaned_seq)

        # Basic properties
        seq_length = len(cleaned_seq)
        mol_weight = analysed_seq.molecular_weight()
        pI = analysed_seq.isoelectric_point()
        instability_index = analysed_seq.instability_index()
        aromaticity = analysed_seq.aromaticity()
        gravy = analysed_seq.gravy()

        # Calculate aliphatic index
        amino_acid_count = analysed_seq.count_amino_acids()
        ala = amino_acid_count.get('A', 0) / seq_length * 100
        val = amino_acid_count.get('V', 0) / seq_length * 100
        ile = amino_acid_count.get('I', 0) / seq_length * 100
        leu = amino_acid_count.get('L', 0) / seq_length * 100
        aliphatic_index = ala + (2.9 * val) + (3.9 * (ile + leu))

        # Amino acid percentages
        amino_acid_percent = analysed_seq.amino_acids_percent
        amino_acid_percent_values = [
            amino_acid_percent.get(aa, 0) * 100 for aa in amino_acids
        ]

        # Secondary structure fraction
        helix, turn, sheet = analysed_seq.secondary_structure_fraction()

        # Extract retained metadata values
        metadata_values = [row[col] for col in metadata_cols]

        # Write data to the CSV file
        writer.writerow(
            metadata_values + [
                cleaned_seq,
                seq_length,
                f"{mol_weight:.2f}",
                f"{pI:.2f}",
                f"{instability_index:.2f}",
                f"{aromaticity:.4f}",
                f"{gravy:.4f}",
                f"{aliphatic_index:.2f}",
                f"{helix:.4f}",
                f"{turn:.4f}",
                f"{sheet:.4f}"
            ] + [f"{value:.2f}" for value in amino_acid_percent_values]
        )

print(f"Saved physicochemical property results to: {csv_file}")


Saved physicochemical property results to: physicochemical_properties.csv


**Prepare sequences for AllerCatPro 2.0**

In [9]:
# ============================================================
# This step:
# 1. Loads the AI prediction output spreadsheet
# 2. Reads sequences directly from the "sequence" column
# 3. Uses the "annotation" column as the FASTA header
# 4. Builds AllerCatPro-compatible FASTA text in memory
# 5. Splits the records into batches of 10 sequences
# ============================================================

import re
import math
import pandas as pd

input_file = "prediction_output.xlsx"
sequences_per_batch = 10

# Load dataset
dataset = pd.read_excel(input_file, na_filter=False)

# Ensure required columns are present
required_cols = ["sequence", "annotation"]
for col in required_cols:
    if col not in dataset.columns:
        raise ValueError(f"The input file must contain a '{col}' column.")

def clean_sequence(seq):
    """Convert sequence to uppercase and remove spaces/newlines."""
    return re.sub(r"\s+", "", str(seq).upper())

def clean_header(text):
    """Clean FASTA header text while retaining readability."""
    text = str(text).strip()
    text = re.sub(r"[\r\n]+", " ", text)
    return text

def wrap_sequence(seq, width=80):
    """Wrap sequence lines to standard FASTA width."""
    return "\n".join(seq[i:i+width] for i in range(0, len(seq), width))

# Build FASTA records
fasta_records = []

for idx, row in dataset.iterrows():
    seq = clean_sequence(row["sequence"])
    header = clean_header(row["annotation"])

    # Skip missing or empty sequences
    if seq == "" or seq.lower() == "nan":
        continue

    # Add sequence index to improve traceability
    fasta_record = f">seq_{idx+1} | {header}\n{wrap_sequence(seq)}"
    fasta_records.append(fasta_record)

if len(fasta_records) == 0:
    raise ValueError("No valid sequences were found in the input file.")

# Split into batches
batches = []
total_batches = math.ceil(len(fasta_records) / sequences_per_batch)

for i in range(total_batches):
    start = i * sequences_per_batch
    end = start + sequences_per_batch
    batch_text = "\n".join(fasta_records[start:end])
    batches.append(batch_text)

print(f"Total valid sequences prepared: {len(fasta_records)}")
print(f"Total batches created: {len(batches)}")

Total valid sequences prepared: 10
Total batches created: 1


# REQUIREMENTS:
# - Google Chrome must already be installed and working
# - Install dependencies first:
#     !pip install selenium webdriver-manager


In [10]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install -q selenium webdriver-manager

Get:1 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:6 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,211 B]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,953 B in 4s (1,952 B/s)
Reading package lis

In [11]:
!which google-chrome
!google-chrome --version

/usr/bin/google-chrome
Google Chrome 147.0.7727.55 


**Submit sequences in batches and download results**

In [12]:
# ============================================================
# This step:
# 1. Opens AllerCatPro
# 2. Pastes FASTA text batches directly into the sequence box
# 3. Submits one batch at a time
# 4. Downloads the resulting CSV files
# 5. Retries failed batches automatically
# 6. Merges all downloaded result files
#
# REQUIREMENTS:
# - Google Chrome must already be installed and working
# - Install dependencies first:
#     !pip install selenium webdriver-manager
# ============================================================

import os
import time
import shutil
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------
url = "https://allercatpro.bii.a-star.edu.sg/"
download_folder = os.path.expanduser("~/Downloads")
output_folder = "./allercatpro_results"
merged_file = os.path.join(output_folder, "merged_allercatpro_results.csv")

# Timing / retry controls
page_load_wait = 3
download_wait = 12
between_batch_wait = 5
max_retries = 2

# ------------------------------------------------------------
# Prepare folders
# ------------------------------------------------------------
os.makedirs(output_folder, exist_ok=True)
os.makedirs(download_folder, exist_ok=True)

# ------------------------------------------------------------
# Clean old CSV files from Downloads to avoid mixing files
# ------------------------------------------------------------
for f in os.listdir(download_folder):
    if f.endswith(".csv"):
        try:
            os.remove(os.path.join(download_folder, f))
        except Exception as e:
            print(f"Warning: could not remove old download file {f}: {e}")

# ------------------------------------------------------------
# Configure Chrome
# ------------------------------------------------------------
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.binary_location = "/usr/bin/google-chrome"

# Enable automatic downloads
prefs = {
    "download.default_directory": os.path.abspath(download_folder),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)

# ------------------------------------------------------------
# Start browser
# ------------------------------------------------------------
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 90)

# ------------------------------------------------------------
# Check batches
# ------------------------------------------------------------
if "batches" not in globals():
    raise NameError(
        "The variable `batches` is not defined. Please create `batches` "
        "as a list of FASTA-formatted text blocks before running this script."
    )

if not isinstance(batches, list) or len(batches) == 0:
    raise ValueError(
        "`batches` must be a non-empty list of FASTA-formatted strings."
    )

# ------------------------------------------------------------
# Helper: find newest downloaded CSV
# ------------------------------------------------------------
def get_latest_csv(download_dir: str) -> str | None:
    csv_files = [
        os.path.join(download_dir, f)
        for f in os.listdir(download_dir)
        if f.endswith(".csv")
    ]
    if not csv_files:
        return None
    return max(csv_files, key=os.path.getctime)

# ------------------------------------------------------------
# Process all batches
# ------------------------------------------------------------
failed_batches = []
completed_batches = []

try:
    for i, batch_text in enumerate(batches, start=1):
        print(f"\nProcessing batch {i}/{len(batches)}")
        success = False

        for attempt in range(1, max_retries + 1):
            try:
                print(f"  Attempt {attempt}/{max_retries}")

                # Open page
                driver.get(url)
                time.sleep(page_load_wait)

                # Locate sequence box
                seq_box = wait.until(
                    EC.presence_of_element_located((By.TAG_NAME, "textarea"))
                )
                seq_box.clear()
                seq_box.send_keys(batch_text)
                print(f"  Pasted batch {i}")

                # Submit batch
                submit_button = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//input[@value='Submit']"))
                )
                submit_button.click()
                print(f"  Submitted batch {i}")

                # Wait for download link
                download_link = wait.until(
                    EC.presence_of_element_located(
                        (By.PARTIAL_LINK_TEXT, "Download result as comma-separated table")
                    )
                )

                driver.execute_script("arguments[0].scrollIntoView();", download_link)
                download_link.click()
                print(f"  Clicked download for batch {i}")

                # Wait for download to complete
                time.sleep(download_wait)

                latest_file = get_latest_csv(download_folder)
                if latest_file is None:
                    raise FileNotFoundError("No CSV file found in Downloads.")

                new_name = os.path.join(
                    output_folder,
                    f"allercatpro_batch_{i}_result.csv"
                )

                # If output file already exists, remove it before moving
                if os.path.exists(new_name):
                    os.remove(new_name)

                shutil.move(latest_file, new_name)
                print(f"  Moved file to: {new_name}")

                completed_batches.append(i)
                success = True
                break

            except Exception as e:
                print(f"  Error in batch {i}, attempt {attempt}: {e}")
                time.sleep(5)

        if not success:
            failed_batches.append(i)
            print(f"  Batch {i} failed after {max_retries} attempts.")

        # Pause between batches to reduce server-side issues
        if i < len(batches):
            time.sleep(between_batch_wait)

finally:
    driver.quit()

# ------------------------------------------------------------
# Merge all downloaded CSV files
# ------------------------------------------------------------
merged_data = pd.DataFrame()

for csv_file in sorted(os.listdir(output_folder)):
    if csv_file.endswith(".csv") and csv_file != os.path.basename(merged_file):
        file_path = os.path.join(output_folder, csv_file)
        try:
            df = pd.read_csv(file_path)
            df["batch_file"] = csv_file
            merged_data = pd.concat([merged_data, df], ignore_index=True)
        except Exception as e:
            print(f"Warning: could not read {csv_file}: {e}")

if not merged_data.empty:
    merged_data.to_csv(merged_file, index=False)
    print(f"\nMerged results saved to: {merged_file}")
    print(f"Merged rows: {merged_data.shape[0]}")
    print(f"Merged columns: {merged_data.shape[1]}")
else:
    print("\nNo CSV files were merged. Please check whether downloads completed successfully.")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\nSummary")
print("-------")
print(f"Total batches      : {len(batches)}")
print(f"Completed batches  : {len(completed_batches)} -> {completed_batches}")
print(f"Failed batches     : {len(failed_batches)} -> {failed_batches}")


Processing batch 1/1
  Attempt 1/2
  Pasted batch 1
  Submitted batch 1
  Clicked download for batch 1
  Moved file to: ./allercatpro_results/allercatpro_batch_1_result.csv

Merged results saved to: ./allercatpro_results/merged_allercatpro_results.csv
Merged rows: 10
Merged columns: 20

Summary
-------
Total batches      : 1
Completed batches  : 1 -> [1]
Failed batches     : 0 -> []
